# Анализ результатов тренажёра

Notebook на C# читает статистику, сохранённую приложением, и сравнивает результаты по текстам. Откройте его в VS Code с расширением Polyglot Notebooks и выберите ядро .NET (C#). Завершите хотя бы одну попытку в тренажёре, затем выполните ячейки по порядку.

Ошибки здесь означают неверные символы в итоговом тексте. Исправленные опечатки в этот показатель не входят. Скорость измеряется в правильно набранных знаках в минуту.


In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Text.Json;

var dataPath = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.LocalApplicationData), "TypingTrainer", "statistics.json");
using var statisticsDocument = JsonDocument.Parse(File.Exists(dataPath) ? File.ReadAllText(dataPath) : "[]");
var sessions = statisticsDocument.RootElement.EnumerateArray().Select(item => new
{
    Date = item.GetProperty("Date").GetDateTime(),
    Text = item.GetProperty("DictionaryName").GetString() ?? "",
    Speed = item.GetProperty("WordsPerMinute").GetDouble(),
    Accuracy = item.GetProperty("Accuracy").GetDouble(),
    Errors = item.GetProperty("Errors").GetInt32()
}).ToArray();
Console.WriteLine($"Завершённых попыток: {sessions.Length}");
Console.WriteLine($"Источник: {dataPath}");


## Общие показатели
Историческое имя поля `WordsPerMinute` в JSON сохранено для совместимости. Приложение записывает в него знаки в минуту.


In [ ]:
if (sessions.Length == 0)
{
    Console.WriteLine("Пока нет результатов. Завершите попытку и повторно выполните ячейки.");
}
else
{
    Console.WriteLine($"Лучшая скорость: {sessions.Max(s => s.Speed):F1} зн/мин");
    Console.WriteLine($"Средняя точность: {sessions.Average(s => s.Accuracy):F1}%");
    Console.WriteLine($"Всего ошибок в завершённых текстах: {sessions.Sum(s => s.Errors)}");
}


In [ ]:
foreach (var group in sessions.GroupBy(s => s.Text).OrderBy(g => g.Key))
{
    Console.WriteLine($"{group.Key}: попыток {group.Count()}, средняя скорость {group.Average(s => s.Speed):F1} зн/мин, точность {group.Average(s => s.Accuracy):F1}%");
}
